# Taylor Root Prediction — Reviewer Demo Notebook

This notebook is designed so that reviewers can reproduce everything **after `git clone` using only relative paths**.

1. (Optional) Install required packages  
2. Generate datasets (for root regressors / for the interval Transformer)  
3. (Optional) Train models (ANN / LSTM / Anchored MLP / Transformer interval predictor)  
4. Run evaluation (includes the baseline solver, failure concentration by func_id, plots)

> ⚠️ Defaults are set for **quick reproduction (~20k samples)**.
> If you have more time/resources, increase `N_TOTAL_*`.


In [ ]:
# (Optional) Install minimal dependencies
# - If your environment is already prepared, you can skip this cell.
#
# !pip install -U pyyaml numpy torch tqdm matplotlib sympy requests
#
# Notes:
# - For PyTorch, you may need a wheel that matches your CUDA/CPU environment.


## 0) Shared utilities / auto-detect repository root

In [ ]:
from __future__ import annotations

from pathlib import Path
import os, sys, subprocess, shutil
import re
import json
import numpy as np

def find_repo_root(start: Path | None = None) -> Path:
# [EN] """   , configs/     ."""
    if start is None:
        start = Path.cwd().resolve()
    else:
        start = start.resolve()

    for p in [start] + list(start.parents):
        if (p / "configs").is_dir():
            return p
    raise RuntimeError("Could not find repo root (missing configs/). Run this notebook from inside the repo.")

REPO = find_repo_root()
print("REPO_ROOT =", REPO)

def R(rel: str | os.PathLike | None) -> Path | None:
# [EN] """    """
    if rel is None:
        return None
    s = str(rel).strip()
    if not s:
        return None
    s = os.path.expanduser(os.path.expandvars(s))
    p = Path(s)
    if p.is_absolute():
        return p
    return (REPO / p).resolve()

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def run(cmd, env=None, cwd=None):
# [EN] """subprocess  + stdout/stderr """
    print(">>", " ".join(map(str, cmd)))
    r = subprocess.run(cmd, env=env, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if r.stdout:
        print("----- STDOUT -----")
        print(r.stdout)
    if r.stderr:
        print("----- STDERR -----")
        print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed (code={r.returncode})")
    return r

def find_generator(keywords: list[str], search_dirs: list[str] | None = None) -> Path | None:
# [EN] """  '  '  """
    if search_dirs is None:
        search_dirs = ["data_generation", "scripts", "tools", "datasets", "dataset", "data", "models", "."]
    roots = [R(d) for d in search_dirs]
    roots = [p for p in roots if p is not None and p.exists()]

    kws = [k.lower() for k in keywords]
    for root in roots:
        for py in root.rglob("*.py"):
            try:
                txt = py.read_text(encoding="utf-8", errors="ignore")
            except Exception:
                continue
            low = txt.lower()
            if all(k in low for k in kws):
                return py
    return None

def migrate_dataset_dir(expected_dir: Path, candidates: list[Path], marker_files: list[str]) -> bool:
# [EN] """    data/...   """
    ensure_dir(expected_dir.parent)
    expected_has = all((expected_dir / f).exists() for f in marker_files)

    if expected_has:
        return False

    for cand in candidates:
        if cand.exists() and all((cand / f).exists() for f in marker_files):
            print(f"[MIGRATE] Found dataset at {cand.relative_to(REPO) if cand.is_relative_to(REPO) else cand}")
            print(f"          -> moving to {expected_dir.relative_to(REPO)}")
            if expected_dir.exists():
                # merge move (if empty) else raise
                if any(expected_dir.iterdir()):
                    raise RuntimeError(f"Expected dir already exists and not empty: {expected_dir}")
                expected_dir.rmdir()
            shutil.move(str(cand), str(expected_dir))
            return True
    return False


## 1) Configure paths + (optional) auto-fix misplaced dataset folders

In [ ]:
# ---- Reviewer defaults (edit if needed) ----
DEGREE = 25

# For quick reproduction: ~20k–100k samples recommended (adjust to your environment).
N_TOTAL_ROOT = 20000
N_TOTAL_INTERVAL = 20000

# [EN] #    ()
OUT_ROOT_DIR = R("data/taylor_data_physchem_v4_deg25")
OUT_INTERVAL_DIR = R("data/taylor_data_physchem_v4_interval")

ensure_dir(OUT_ROOT_DIR)
ensure_dir(OUT_INTERVAL_DIR)

# Root-regression NPZ (ann/lstm/mlp)
train_npz = OUT_ROOT_DIR / f"taylor_deg{DEGREE}_train.npz"
val_npz   = OUT_ROOT_DIR / f"taylor_deg{DEGREE}_val.npz"
test_npz  = OUT_ROOT_DIR / f"taylor_deg{DEGREE}_test.npz"

# Interval(Transformer) NPZ
interval_train_npz = OUT_INTERVAL_DIR / f"taylor_deg{DEGREE}_train.npz"
interval_val_npz   = OUT_INTERVAL_DIR / f"taylor_deg{DEGREE}_val.npz"
interval_test_npz  = OUT_INTERVAL_DIR / f"taylor_deg{DEGREE}_test.npz"

# [EN] # ---- " "   ----
# [EN] #  / data/    , root/    .
migrate_dataset_dir(
    expected_dir=OUT_ROOT_DIR,
    candidates=[R("taylor_data_physchem_v4_deg25"), R("root/taylor_data_physchem_v4_deg25")],
    marker_files=[f"taylor_deg{DEGREE}_train.npz", f"taylor_deg{DEGREE}_val.npz", f"taylor_deg{DEGREE}_test.npz"],
)

migrate_dataset_dir(
    expected_dir=OUT_INTERVAL_DIR,
    candidates=[R("taylor_data_physchem_v4_interval"), R("root/taylor_data_physchem_v4_interval")],
    marker_files=[f"taylor_deg{DEGREE}_train.npz", f"taylor_deg{DEGREE}_val.npz", f"taylor_deg{DEGREE}_test.npz"],
)

print("[ROOT NPZ paths]")
print(" train:", train_npz.relative_to(REPO))
print(" val  :", val_npz.relative_to(REPO))
print(" test :", test_npz.relative_to(REPO))

print("\n[INTERVAL NPZ paths]")
print(" train:", interval_train_npz.relative_to(REPO))
print(" val  :", interval_val_npz.relative_to(REPO))
print(" test :", interval_test_npz.relative_to(REPO))


## 2) Auto-discover dataset generator scripts

In [ ]:
# ============================
# [EN] # Dataset generator   (⚠️ models/   )
# [EN] # -   models/transformer/model.py  " "
# [EN] #      , scripts/   .
# ============================

from pathlib import Path

def find_generator_scoped(
    *,
    keywords: list[str],
    search_dirs: list[Path],
    prefer_name_keywords: list[str] | None = None,
    exclude_substrings: list[str] | None = None,
    max_show: int = 8,
) -> Path | None:
    prefer_name_keywords = prefer_name_keywords or []
    exclude_substrings = exclude_substrings or []

    cand_files: list[Path] = []
    for d in search_dirs:
        if d.exists():
            cand_files.extend(list(d.rglob("*.py")))

# [EN] # fallback: repo
    if not cand_files:
        cand_files = list(REPO.rglob("*.py"))

    scored = []
    for p in cand_files:
        rel = p.relative_to(REPO)
        rel_s = str(rel).replace("\\", "/")

        # exclude
        if any(x in rel_s for x in exclude_substrings):
            continue
        if rel_s.startswith("models/"):
            continue
        if rel_s.startswith(".") or "/.venv/" in rel_s or "/site-packages/" in rel_s:
            continue

        try:
            txt = p.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue

        score = 0
        for k in keywords:
            if k in txt:
                score += 3
            if k in rel_s:
                score += 2

        for k in prefer_name_keywords:
            if k in rel_s.lower():
                score += 2

# [EN] # : argparse
        if "--out-dir" in txt or "np.savez" in txt or "templates.json" in txt:
            score += 1

        if score <= 0:
            continue

        scored.append((score, p))

    if not scored:
        return None

    scored.sort(key=lambda t: (-t[0], str(t[1])))
    top = scored[:max_show]
    print("[GENERATOR CANDIDATES]")
    for s, p in top:
        print(f"  score={s:3d}  {p.relative_to(REPO)}")

    return top[0][1]


# [EN] # root-regressor  (ann/lstm/mlp )
# [EN] # - templates.json , 'dataset generation finished'
gen_root = find_generator_scoped(
    keywords=["templates.json", "dataset generation finished", "generate_dataset"],
    search_dirs=[REPO/"scripts", REPO/"data_generation", REPO/"tools", REPO/"scripts/data"],
    prefer_name_keywords=["deg25", "physchem", "v4"],
    exclude_substrings=["models/"],
)

# [EN] # transformer interval
# [EN] # - interval  expr_str / roots     +  interval
gen_interval = find_generator_scoped(
    keywords=["expr_str", "roots", "taylor_deg", "np.savez"],
    search_dirs=[REPO/"scripts", REPO/"data_generation", REPO/"tools", REPO/"scripts/data"],
    prefer_name_keywords=["interval"],
    exclude_substrings=["models/"],
)

print("\n[FOUND GENERATORS]")
print(" root_gen     =", (gen_root.relative_to(REPO) if gen_root else None))
print(" interval_gen =", (gen_interval.relative_to(REPO) if gen_interval else None))

if gen_root is None:
    raise RuntimeError(
        "Root dataset generator script not found.\n"
# [EN] "→ scripts/data/  root  (.py)   ."
    )

if gen_interval is None:
    raise RuntimeError(
        "Interval dataset generator script not found.\n"
# [EN] "→ scripts/data/  interval  (.py)   .\n"
# [EN] "   ( : models/transformer/model.py  ,    .)"
    )

## 3) Generate the root-regression dataset (ANN / LSTM / Anchored MLP)

In [ ]:
need_root = (not train_npz.exists()) or (not val_npz.exists()) or (not test_npz.exists())

if not need_root:
    print("[SKIP] root npz already exists.")
else:
    cmd = [
        sys.executable, str(gen_root),
        "--degree", str(DEGREE),
        "--n-total", str(N_TOTAL_ROOT),
        "--seed", "42",
# [EN] "--out-dir", os.path.relpath(str(OUT_ROOT_DIR), str(REPO)),  #
        "--save-expr-str", "1",
    ]
    run(cmd, env={"PYTHONPATH": str(REPO), **os.environ}, cwd=str(REPO))

print("[ROOT DATA CHECK]")
for p in [train_npz, val_npz, test_npz]:
    print(" ", p.relative_to(REPO), "exists=", p.exists())


## 4) Generate the interval (Transformer) dataset

In [ ]:
need_interval = (not interval_train_npz.exists()) or (not interval_val_npz.exists()) or (not interval_test_npz.exists())

if not need_interval:
    print("[SKIP] interval npz already exists.")
else:
    cmd = [
        sys.executable, str(gen_interval),
        "--degree", str(DEGREE),
        "--n-total", str(N_TOTAL_INTERVAL),
        "--seed", "42",
# [EN] "--out-dir", os.path.relpath(str(OUT_INTERVAL_DIR), str(REPO)),  #
        "--save-expr-str", "1",
    ]
    run(cmd, env={"PYTHONPATH": str(REPO), **os.environ}, cwd=str(REPO))

print("[INTERVAL DATA CHECK]")
for p in [interval_train_npz, interval_val_npz, interval_test_npz]:
    print(" ", p.relative_to(REPO), "exists=", p.exists())


# safety: gen_interval must not be the training script
assert 'models/transformer/model.py' not in str(gen_interval).replace('\\','/'), f"gen_interval points to training code: {gen_interval}"


## 5) (Optional) Quick sanity-check dataset keys/shapes

In [ ]:
import numpy as np

def peek_npz(npz_path: Path, keys_top: int = 20):
    z = np.load(npz_path, allow_pickle=True)
    keys = list(z.keys())
    print(f"[NPZ] {npz_path.relative_to(REPO)}")
    print(" keys:", keys[:keys_top], ("..." if len(keys)>keys_top else ""))
    for k in keys[:min(8, len(keys))]:
        v = z[k]
        try:
            shape = v.shape
        except Exception:
            shape = "?"
        print(f"  - {k:12s} dtype={getattr(v,'dtype',None)} shape={shape}")
    print()

peek_npz(train_npz)
peek_npz(interval_train_npz)


## 6) (Optional) Train models
- Disabled by default (commented out).
- When enabled, outputs will be written under `results/`.
- Uses **only relative paths**.

In [ ]:
import torch

def train_script(script_rel: str, cfg_rel: str, out_rel: str,
                 train_path: Path, val_path: Path, test_path: Path | None,
                 extra_env: dict | None = None):
    script = R(script_rel)
    assert script and script.exists(), f"Script not found: {script_rel}"
    env = os.environ.copy()
    env.update({
# [EN] "TAYLOR_CFG": str(R(cfg_rel)),                 #  ->
        "TRAIN_NPZ": str(train_path),
        "VAL_NPZ": str(val_path),
        "TEST_NPZ": (str(test_path) if test_path is not None else ""),
        "OUT_DIR": str(R(out_rel)),
        "DEVICE": ("cuda" if torch.cuda.is_available() else "cpu"),
        "PYTHONPATH": str(REPO),
    })
    if extra_env:
        env.update({k: str(v) for k,v in extra_env.items()})
    ensure_dir(R(out_rel))
    run([sys.executable, str(script)], env=env, cwd=str(REPO))

# ---- Uncomment only what you want to run ----

# train_script("models/taylor_nn/ann.py", "configs/taylor_root_ann.yaml", "results/taylor_nn/ann",
#              train_npz, val_npz, test_npz)

# train_script("models/taylor_nn/lstm.py", "configs/taylor_root_lstm.yaml", "results/taylor_nn/lstm",
#              train_npz, val_npz, test_npz)

# train_script("models/taylor_nn/mlp.py", "configs/taylor_root_mlp.yaml", "results/taylor_nn/mlp",
#              train_npz, val_npz, test_npz)

# train_script("models/transformer/model.py", "configs/transformer_interval.yaml", "results/transformer_interval",
#              interval_train_npz, interval_val_npz, interval_test_npz, extra_env={"MODE":"train"})


## 7) Run evaluation (includes baseline)
- `evaluation/evaluate_k_sweep.py` + `configs/eval_k_sweep.yaml`
- Failure concentration by func_id, histograms, and boxplots are controlled via YAML toggles.

In [ ]:
import torch
from pathlib import Path

eval_script = R("evaluation/evaluate_k_sweep.py")
assert eval_script and eval_script.exists(), f"Missing eval script: {eval_script}"

# Output directory (relative path)
OUTDIR = R("results/runs_k_sweep_viz")
ensure_dir(OUTDIR)

env = os.environ.copy()
env["EVAL_CFG"]  = str(R("configs/eval_k_sweep.yaml"))
env["OUTDIR"]    = str(OUTDIR)
env["DEVICE"]    = ("cuda" if torch.cuda.is_available() else "cpu")
env["PYTHONPATH"]= str(REPO)

run([sys.executable, str(eval_script)], env=env, cwd=str(REPO))

print("\n[OK] Evaluation finished.")
print("Outputs under:", OUTDIR.relative_to(REPO))


## 8) Quick check of generated outputs (tables/figures/reports)

In [ ]:
from glob import glob

outdir = R("results/runs_k_sweep_viz")
pngs = sorted(glob(str(outdir / "*.png")))
csvs = sorted(glob(str(outdir / "*.csv")))
jsons = sorted(glob(str(outdir / "*.json")))

print("[PNG] ", len(pngs))
for p in pngs[:12]:
    print(" ", Path(p).relative_to(REPO))
if len(pngs) > 12:
    print("  ...")

print("\n[CSV] ", len(csvs))
for p in csvs[:12]:
    print(" ", Path(p).relative_to(REPO))
if len(csvs) > 12:
    print("  ...")

print("\n[JSON] ", len(jsons))
for p in jsons[:12]:
    print(" ", Path(p).relative_to(REPO))
if len(jsons) > 12:
    print("  ...")


## 9) (Optional) Inspect failure concentration for baseline/models (by func_id)
- Reads the JSON/CSV produced when `reports.report_fail_funcid: true` in `eval_k_sweep.yaml`.

In [ ]:
import pandas as pd

# Auto-find one of the saved fail_by_funcid_*.csv files
outdir = R("results/runs_k_sweep_viz")
cands = sorted(outdir.glob("fail_by_funcid_*.csv"))

if not cands:
# [EN] print("[INFO] fail_by_funcid_*.csv not found. (reports.report_fail_funcid YAML  )")
else:
    p = cands[0]
    print("[LOAD]", p.relative_to(REPO))
    df = pd.read_csv(p)
    display(df.head(20))
